In [1]:
import pandas as pd
import numpy as np
import kagglehub

In [2]:
path = kagglehub.dataset_download('prithwirajmitra/covid-face-mask-detection-dataset')

In [3]:
import os
train_path = os.path.join(path, 'New Masks Dataset', 'Train')

In [4]:
test_path = os.path.join(path, 'New Masks Dataset', 'Test')

In [5]:
val_path = os.path.join(path, 'New Masks Dataset', 'Validation')

In [6]:
image_height = 224
image_width = 224
batch_size = 32

In [7]:
import tensorflow as tf

In [8]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    color_mode='rgb',
    image_size=(image_height, image_width),
    batch_size=batch_size
)

Found 600 files belonging to 2 classes.


In [9]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    color_mode='rgb',
    image_size=(image_height, image_width),
    batch_size=batch_size
)

Found 100 files belonging to 2 classes.


In [10]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_path,
    color_mode='rgb',
    image_size=(image_height, image_width),
    batch_size=batch_size
)

Found 306 files belonging to 2 classes.


In [11]:
train_ds.class_names

['Mask', 'Non Mask']

In [12]:
val_ds.class_names

['Mask', 'Non Mask']

In [13]:
test_ds.class_names

['Mask', 'Non Mask']

In [14]:
def normalize(image, label):
    image = tf.cast(image/255, tf.float32)
    return image, label

In [15]:
train_ds = train_ds.map(normalize)
test_ds = test_ds.map(normalize)

In [16]:
val_ds = val_ds.map(normalize)

In [17]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomFlip('horizontal')
])

In [18]:
def augment(image,label):
    image = data_augmentation(image)
    return image, label

In [19]:
train_ds_augment = train_ds.map(augment)

In [20]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(image_height, image_width, 3)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

D:\Projects for Internship\Face Mask Detection\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [22]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 222, 222, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 111, 111, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 109, 109, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 54, 54, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 52, 52, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 26, 26, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 86528)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │      11,075,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
epochs = 20

early_stopping = tf.keras.callbacks.EarlyStopping(
    patience=5,
    monitor='val_loss',
    restore_best_weights=True
)

In [24]:
history = model.fit(train_ds_augment, validation_data=val_ds, batch_size=32, callbacks=early_stopping, epochs=epochs)

D:\Projects for Internship\Face Mask Detection\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 12s 544ms/step - accuracy: 0.5333 - loss: 1.1006 - val_accuracy: 0.7059 - val_loss: 0.5615
Epoch 2/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 10s 516ms/step - accuracy: 0.8450 - loss: 0.4169 - val_accuracy: 0.8889 - val_loss: 0.2871
Epoch 3/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 10s 528ms/step - accuracy: 0.9150 - loss: 0.2628 - val_accuracy: 0.8824 - val_loss: 0.3071
Epoch 4/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 10s 522ms/step - accuracy: 0.9150 - loss: 0.2487 - val_accuracy: 0.9052 - val_loss: 0.2759
Epoch 5/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 10s 531ms/step - accuracy: 0.9117 - loss: 0.2309 - val_accuracy: 0.8725 - val_loss: 0.2728
Epoch 6/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 10s 530ms/step - accuracy: 0.9217 - loss: 0.2076 - val_accuracy: 0.8693 - val_loss: 0.3367
Epoch 7/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 10s 525ms/step - accuracy: 0.9133 - loss: 0.2154 - val_accuracy: 0.9020 - val_loss: 0.2486
Epoch 8/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 10s 522ms/step - accuracy: 0.9117 - loss: 0.2278 - val_accu

In [25]:
test_loss, test_accuracy = model.evaluate(test_ds)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.9300 - loss: 0.1575


In [32]:
model.save('face_mask_detection_model.keras')

In [33]:
import pickle

In [34]:
pickle.dump(image_height, open('image_height.pkl','wb'))
pickle.dump(image_width, open('image_width.pkl','wb'))
pickle.dump(batch_size, open('batch_size.pkl','wb'))